In [ ]:
import os
import glob
import pandas as pd
import kagglehub
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

TARGET_LABELS = [
    'Angry', 'Anxiety', 'Disgust', 'Excitement',
    'Fear', 'Happiness', 'Sadness', 'Sleepiness', 'Surprise'
]
label2id = {label: i for i, label in enumerate(TARGET_LABELS)}
id2label = {i: label for i, label in enumerate(TARGET_LABELS)}

# Target map for unifying multi-dataset taxonomies (GoEmotions & DAIR.AI) to our 9 categories
unified_mapping = {
    'anger': 'Angry', 'angry': 'Angry', 'annoyance': 'Angry', 'disapproval': 'Angry',
    'nervousness': 'Anxiety', 'anxiety': 'Anxiety',
    'disgust': 'Disgust',
    'excitement': 'Excitement',
    'fear': 'Fear',
    'joy': 'Happiness', 'amusement': 'Happiness', 'approval': 'Happiness', 'happy': 'Happiness',
    'sadness': 'Sadness', 'grief': 'Sadness', 'disappointment': 'Sadness',
    'tired': 'Sleepiness', 'fatigue': 'Sleepiness', 'sleepiness': 'Sleepiness',
    'surprise': 'Surprise', 'realization': 'Surprise'
}

# Pull dependencies via KaggleHub resolver
goemotions_path = kagglehub.dataset_download("debarshichanda/goemotions")
dair_path = kagglehub.dataset_download("fazalkhan09/dataset")

: 

In [ ]:
# Re-define labels for 3-class Sentiment Analysis
import os
import glob
import pandas as pd
from sklearn.model_selection import train_test_split
import kagglehub

# Ensure paths are resolved
goemotions_path = kagglehub.dataset_download("debarshichanda/goemotions")
dair_path = kagglehub.dataset_download("fazalkhan09/dataset")

SENTIMENT_LABELS = ['Negative', 'Neutral', 'Positive']
sent_label2id = {label: i for i, label in enumerate(SENTIMENT_LABELS)}
sent_id2label = {i: label for i, label in enumerate(SENTIMENT_LABELS)}

# Map emotion categories to 3 main sentiment labels
emotion_to_sentiment = {
    'Angry': 'Negative',
    'Anxiety': 'Negative',
    'Disgust': 'Negative',
    'Fear': 'Negative',
    'Sadness': 'Negative',
    'Sleepiness': 'Neutral',
    'Surprise': 'Neutral',
    'Excitement': 'Positive',
    'Happiness': 'Positive'
}

# Adapt standardization to output sentiment labels
def load_and_standardize_sentiment(path, text_col, label_col=None):
    data_files = (
        glob.glob(os.path.join(path, "**", "*.csv"), recursive=True) +
        glob.glob(os.path.join(path, "**", "*.parquet"), recursive=True) +
        glob.glob(os.path.join(path, "**", "*.txt"), recursive=True) +
        glob.glob(os.path.join(path, "**", "*.pkl"), recursive=True)
    )
    if not data_files:
        raise FileNotFoundError(f"No matching datasets discovered under directory: {path}")

    data_files.sort(key=os.path.getsize, reverse=True)
    target_file = data_files[0]

    if target_file.endswith('.parquet'):
        df = pd.read_parquet(target_file)
    elif target_file.endswith('.pkl'):
        df = pd.read_pickle(target_file)
        if isinstance(df, dict):
            df = pd.concat([pd.DataFrame(v) for v in df.values()], ignore_index=True)
    elif target_file.endswith('.txt'):
        df = pd.read_csv(target_file, sep=';', names=['text', 'label'] if label_col is None else [text_col, label_col])
    else:
        df = pd.read_csv(target_file)

    if text_col not in df.columns:
        possible_text_cols = ['text', 'sentence', 'text_data', 'sentences', 'document']
        found_text_col = next((c for c in possible_text_cols if c in df.columns), None)
        text_col = found_text_col if found_text_col else (df.select_dtypes(include=['object']).columns[0] if len(df.select_dtypes(include=['object']).columns) > 0 else df.columns[0])

    detected_label_col = label_col if (label_col is not None and label_col in df.columns) else next((c for c in ['label', 'emotions', 'emotion', 'sentiment', 'category'] if c in df.columns), None)

    if detected_label_col is None:
        available_emotions = [col for col in unified_mapping.keys() if col in df.columns]
        if len(available_emotions) > 0:
            df['emotion_label'] = None
            for emotion in available_emotions:
                df.loc[df[emotion] == 1, 'emotion_label'] = unified_mapping[emotion]
        else:
            raise KeyError(f"Unable to locate target label columns.")
    else:
        df['emotion_label'] = df[detected_label_col].astype(str).str.lower().map(unified_mapping)

    # Rename and clean
    df = df.rename(columns={text_col: 'text'}).dropna(subset=['text', 'emotion_label'])

    # Map Emotion to Sentiment
    df['label'] = df['emotion_label'].map(emotion_to_sentiment)
    df = df.dropna(subset=['label'])
    df['label_id'] = df['label'].map(sent_label2id).astype(int)

    return df[['text', 'label', 'label_id']]

df_goemotions_sent = load_and_standardize_sentiment(goemotions_path, text_col='text')
df_dair_sent = load_and_standardize_sentiment(dair_path, text_col='text', label_col='label')

train_df_sent = df_goemotions_sent.copy()
val_df_sent, test_df_sent = train_test_split(df_dair_sent, test_size=0.5, random_state=42)

In [ ]:
# Initialize Tokenizer and Dataloaders for Sentiment Analysis
from transformers import AutoTokenizer
from torch.utils.data import DataLoader
import torch

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

train_loader_sent = DataLoader(EmotionDataset(train_df_sent, tokenizer), batch_size=32, shuffle=True)
val_loader_sent = DataLoader(EmotionDataset(val_df_sent, tokenizer), batch_size=32)
test_loader_sent = DataLoader(EmotionDataset(test_df_sent, tokenizer), batch_size=32)

In [ ]:
# Configure and instantiate Model with 3 classes
from transformers import AutoModelForSequenceClassification
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model_sent = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(SENTIMENT_LABELS),
    id2label=sent_id2label,
    label2id=sent_label2id
).to(device)

optimizer_sent = torch.optim.AdamW(model_sent.parameters(), lr=5e-5)
criterion_sent = torch.nn.CrossEntropyLoss()

In [ ]:
# Initialize Tokenizer and Dataloaders for Sentiment Analysis
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader
import torch
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Target map for unifying multi-dataset taxonomies (GoEmotions & DAIR.AI) to our 9 categories
unified_mapping = {
    'anger': 'Angry', 'angry': 'Angry', 'annoyance': 'Angry', 'disapproval': 'Angry',
    'nervousness': 'Anxiety', 'anxiety': 'Anxiety',
    'disgust': 'Disgust',
    'excitement': 'Excitement',
    'fear': 'Fear',
    'joy': 'Happiness', 'amusement': 'Happiness', 'approval': 'Happiness', 'happy': 'Happiness',
    'sadness': 'Sadness', 'grief': 'Sadness', 'disappointment': 'Sadness',
    'tired': 'Sleepiness', 'fatigue': 'Sleepiness', 'sleepiness': 'Sleepiness',
    'surprise': 'Surprise', 'realization': 'Surprise'
}

# Ensure dataset variables are defined in this scope
try:
    _ = train_df_sent
except NameError:
    df_goemotions_sent = load_and_standardize_sentiment(goemotions_path, text_col='text')
    df_dair_sent = load_and_standardize_sentiment(dair_path, text_col='text', label_col='label')
    train_df_sent = df_goemotions_sent.copy()
    val_df_sent, test_df_sent = train_test_split(df_dair_sent, test_size=0.5, random_state=42)

# Ensure EmotionDataset is defined
class EmotionDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.texts = df['text'].values
        self.labels = df['label_id'].values
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_loader_sent = DataLoader(EmotionDataset(train_df_sent, tokenizer), batch_size=32, shuffle=True)
val_loader_sent = DataLoader(EmotionDataset(val_df_sent, tokenizer), batch_size=32)
test_loader_sent = DataLoader(EmotionDataset(test_df_sent, tokenizer), batch_size=32)

# Evaluate function definition
def evaluate_sent(model, dataloader, device, criterion):
    model.eval()
    total_loss, correct_preds, total_preds = 0, 0, 0

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating", leave=False):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            total_loss += criterion(logits, labels).item() * len(labels)
            correct_preds += torch.sum(torch.argmax(logits, dim=1) == labels).item()
            total_preds += len(labels)

    return total_loss / total_preds, correct_preds / total_preds

# Fine-tune Model for Sentiment Analysis
print("Starting training loop for Sentiment Analysis...")
model_sent.train()
for epoch in range(1):
    for batch in tqdm(train_loader_sent, desc=f"Epoch {epoch + 1}/1"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer_sent.zero_grad()
        logits = model_sent(input_ids=input_ids, attention_mask=attention_mask).logits
        loss = criterion_sent(logits, labels)
        loss.backward()
        optimizer_sent.step()

    val_loss, val_acc = evaluate_sent(model_sent, val_loader_sent, device, criterion_sent)
    print(f"Epoch finished | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

In [ ]:
# Predict and evaluate on Sentiment classification
def get_predictions_sent(model, dataloader, device):
    model.eval()
    predictions = []
    real_values = []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            predictions.extend(preds)
            real_values.extend(labels.cpu().numpy())
    return np.array(real_values), np.array(predictions)

y_true_sent, y_pred_sent = get_predictions_sent(model_sent, test_loader_sent, device)

# Detailed classification report
present_classes_sent = sorted(list(set(y_true_sent) | set(y_pred_sent)))
class_names_sent = [sent_id2label[i] for i in present_classes_sent]

print("Classification Report:")
print(classification_report(
    y_true_sent,
    y_pred_sent,
    labels=present_classes_sent,
    target_names=class_names_sent,
    zero_division=0
))

In [ ]:
def load_and_standardize_csv(path, text_col, label_col=None):
    # Multi-format resolver for CSV, Parquet, TXT, and PKL formats
    data_files = (
        glob.glob(os.path.join(path, "**", "*.csv"), recursive=True) +
        glob.glob(os.path.join(path, "**", "*.parquet"), recursive=True) +
        glob.glob(os.path.join(path, "**", "*.txt"), recursive=True) +
        glob.glob(os.path.join(path, "**", "*.pkl"), recursive=True)
    )

    if not data_files:
        raise FileNotFoundError(f"No matching datasets discovered under directory: {path}")

    # Pick largest file assuming it is the main serialized table
    data_files.sort(key=os.path.getsize, reverse=True)
    target_file = data_files[0]

    if target_file.endswith('.parquet'):
        df = pd.read_parquet(target_file)
    elif target_file.endswith('.pkl'):
        df = pd.read_pickle(target_file)
        if isinstance(df, dict):
            df = pd.concat([pd.DataFrame(v) for v in df.values()], ignore_index=True)
    elif target_file.endswith('.txt'):
        df = pd.read_csv(target_file, sep=';', names=['text', 'label'] if label_col is None else [text_col, label_col])
    else:
        df = pd.read_csv(target_file)

    # Resolve target text column with fallback heuristical lookups
    if text_col not in df.columns:
        possible_text_cols = ['text', 'sentence', 'text_data', 'sentences', 'document']
        found_text_col = next((c for c in possible_text_cols if c in df.columns), None)
        text_col = found_text_col if found_text_col else (df.select_dtypes(include=['object']).columns[0] if len(df.select_dtypes(include=['object']).columns) > 0 else df.columns[0])

    # Resolve label column (heuristics or mapping binary values to single target category)
    detected_label_col = label_col if (label_col is not None and label_col in df.columns) else next((c for c in ['label', 'emotions', 'emotion', 'sentiment', 'category'] if c in df.columns), None)

    if detected_label_col is None:
        # Melt multi-column binary layout to a standardized categorical label representation
        available_emotions = [col for col in unified_mapping.keys() if col in df.columns]
        if len(available_emotions) > 0:
            df['label'] = None
            for emotion in available_emotions:
                df.loc[df[emotion] == 1, 'label'] = unified_mapping[emotion]
        else:
            raise KeyError(f"Unable to locate target label columns. Available: {list(df.columns)}")
    else:
        df['label'] = df[detected_label_col].astype(str).str.lower().map(unified_mapping)

    # Final serialization alignment
    df = df.rename(columns={text_col: 'text'}).dropna(subset=['text', 'label'])
    df['label_id'] = df['label'].map(label2id).dropna().astype(int)

    return df[['text', 'label', 'label_id']]

In [ ]:
# Load and standardize source structures
df_goemotions = load_and_standardize_csv(goemotions_path, text_col='text')
df_dair = load_and_standardize_csv(dair_path, text_col='text', label_col='label')

# Configure out-of-domain evaluation splits
train_df = df_goemotions.copy()
val_df, test_df = train_test_split(df_dair, test_size=0.5, random_state=42)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(TARGET_LABELS),
    id2label=id2label,
    label2id=label2id
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
criterion = torch.nn.CrossEntropyLoss()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
from tqdm.auto import tqdm
import torch

# Predict on the test dataset loader
y_true, y_pred = get_predictions(model, test_loader, device)

# Calculate and print metrics
accuracy = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)

print(f"=== Test Performance ===")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}\n")

# Identify present classes
present_classes = sorted(list(set(y_true) | set(y_pred)))
class_names = [id2label[i] for i in present_classes]

# Detailed classification report
print("Classification Report:")
print(classification_report(
    y_true,
    y_pred,
    labels=present_classes,
    target_names=class_names,
    zero_division=0
))

# Compute and plot Heat Map of the Confusion Matrix
cm = confusion_matrix(y_true, y_pred, labels=present_classes)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names
)
plt.title('Confusion Matrix - Emotion Classification')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

In [ ]:
# Initialize Tokenizer and Dataloaders
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

class EmotionDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.texts = df['text'].values
        self.labels = df['label_id'].values
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_loader = DataLoader(EmotionDataset(train_df, tokenizer), batch_size=32, shuffle=True)
val_loader = DataLoader(EmotionDataset(val_df, tokenizer), batch_size=32)
test_loader = DataLoader(EmotionDataset(test_df, tokenizer), batch_size=32)

# Evaluate function
def evaluate(model, dataloader, device, criterion):
    model.eval()
    total_loss, correct_preds, total_preds = 0, 0, 0

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating", leave=False):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            total_loss += criterion(logits, labels).item() * len(labels)
            correct_preds += torch.sum(torch.argmax(logits, dim=1) == labels).item()
            total_preds += len(labels)

    return total_loss / total_preds, correct_preds / total_preds

# Training loop
print("Starting training loop...")
model.train()
for epoch in range(1):
    for batch in tqdm(train_loader, desc=f"Epoch {epoch + 1}/1"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

    val_loss, val_acc = evaluate(model, val_loader, device, criterion)
    print(f"Epoch finished | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

In [ ]:
### Dataloader configuration has been merged into the main training pipeline below to maintain order.

In [ ]:
import os

# Run the standard model evaluation with the saved evaluate helper
test_loss, test_accuracy = evaluate(model, test_loader, device, criterion)
print(f"\nTest Loss: {test_loss:.4f} | Test Accuracy: {test_accuracy:.4f}")

# Serialize artifacts
output_dir = "./distilbert_emotion_model"
os.makedirs(output_dir, exist_ok=True)
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Model and tokenizer saved to {output_dir}")